# Experiment 6: CGM + time_since_meal + image (dinov2, gated)

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [2]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.multimodal import MultimodalGRUPredictor

MODEL_NAME = 'CGM+time+dinov2 (gated)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = 'dinov2'

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_initial_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'gated_dinov2'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_dinov2_stability_summary.json'
VAL_STABILITY_SUMMARY_PATH = RESULTS_DIR / 'gated_cgm_time_dinov2_val_stability_summary.json'

In [3]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

val_participant_ids = data.load_multimodal_participant_ids('val')
VAL_PARTICIPANT_IDS = sorted(set(val_participant_ids))

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print()
print('number of test participants:', len(VAL_PARTICIPANT_IDS))
print('Corresponding participants ID: \n', VAL_PARTICIPANT_IDS)
print()
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants: 6
Corresponding participants ID: 
 [np.str_('CGMacros-005'), np.str_('CGMacros-016'), np.str_('CGMacros-027'), np.str_('CGMacros-044'), np.str_('CGMacros-046'), np.str_('CGMacros-048')]


number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 768), (256, 5)]


In [4]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, img_feat, target = batch
    return (model(cgm.to(device), tsm.to(device), img_feat.to(device), has_meal.to(device)),
            target.to(device))

In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []

for seed in SEEDS:
    print(f"Validation results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)
    model = MultimodalGRUPredictor(img_feat_dim=768, **{'gru_hidden_size': 256, 'gru_num_layers': 3, 'gru_dropout': 0.2})

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'gated_dinov2' / f'gated_cgm_time_dinov2_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)

    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    val_results_per_participant = metrics.evaluate_per_participant(
        y_pred_val, y_true_val, val_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'val_results_per_participant': val_results_per_participant,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'gated_cgm_time_dinov2_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = val_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Validation results — CGM+time+dinov2 (gated) | seed=7

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.73 | MAE 8.71 | gRMSE 14.12 | Zone A 94.42% | A+B 99.46%
 30 min | RMSE 18.37 | MAE 12.34 | gRMSE 21.11 | Zone A 88.69% | A+B 99.21%
 60 min | RMSE 25.97 | MAE 17.26 | gRMSE 31.11 | Zone A 80.10% | A+B 98.65%
 90 min | RMSE 30.78 | MAE 20.68 | gRMSE 37.55 | Zone A 74.46% | A+B 98.12%
120 min | RMSE 33.45 | MAE 22.88 | gRMSE 41.13 | Zone A 70.47% | A+B 97.74%

--------------------------------------------------------------------------------
Validation results — CGM+time+dinov2 (gated) | seed=14

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.71 | MAE 7.96 | gRMSE 13.02 | Zone A 96.02% | A+B 99.57%
 30 min | RMSE 18.13 | MAE 12.10 | gRMSE 20.87 | Zone A 88.84% | A+B 99.23%
 60 min | RMSE 26.32 | MAE 17.39 | gRMSE 31.45 | Zone A 79.69% | A+B 98.58%
 90 min | RMSE 31.24 | MAE 20.80 | gRMSE 38.11 | Zone A 74.14% | A+B 98.12%
120 min | RMSE 34.11 | MAE 23.17 | gRMSE 41.93 | Zone A 70.16% | A+B 97.82%

--------------------------------------------------------------------------------
Validation results — CGM+time+dinov2 (gated) | seed=21

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


...

 15 min | RMSE 11.82 | MAE 8.03 | gRMSE 13.01 | Zone A 95.72% | A+B 99.59%
 30 min | RMSE 18.12 | MAE 12.06 | gRMSE 20.59 | Zone A 88.77% | A+B 99.22%
 60 min | RMSE 26.41 | MAE 17.38 | gRMSE 31.31 | Zone A 79.72% | A+B 98.61%
 90 min | RMSE 31.31 | MAE 20.87 | gRMSE 38.00 | Zone A 74.15% | A+B 98.09%
120 min | RMSE 33.85 | MAE 22.85 | gRMSE 41.44 | Zone A 70.72% | A+B 97.82%

--------------------------------------------------------------------------------
Validation results — CGM+time+dinov2 (gated) | seed=63

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.96 | MAE 8.74 | gRMSE 14.11 | Zone A 94.09% | A+B 99.46%
 30 min | RMSE 18.61 | MAE 12.38 | gRMSE 21.02 | Zone A 88.27% | A+B 99.22%
 60 min | RMSE 26.45 | MAE 17.40 | gRMSE 31.41 | Zone A 80.00% | A+B 98.65%
 90 min | RMSE 31.26 | MAE 20.74 | gRMSE 38.22 | Zone A 74.58% | A+B 98.06%
120 min | RMSE 34.05 | MAE 23.02 | gRMSE 41.97 | Zone A 70.49% | A+B 97.69%

--------------------------------------------------------------------------------
Validation results — CGM+time+dinov2 (gated) | seed=70

device: cuda (NVIDIA RTX PRO 6000 Blackwell Server Edition)  model: MultimodalGRUPredictor  params: 1,326,293


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 12.79 | MAE 8.69 | gRMSE 14.17 | Zone A 94.63% | A+B 99.48%
 30 min | RMSE 18.57 | MAE 12.36 | gRMSE 21.28 | Zone A 88.62% | A+B 99.22%
 60 min | RMSE 26.36 | MAE 17.27 | gRMSE 31.52 | Zone A 80.28% | A+B 98.62%
 90 min | RMSE 31.42 | MAE 20.77 | gRMSE 38.34 | Zone A 74.66% | A+B 98.08%
120 min | RMSE 33.88 | MAE 22.97 | gRMSE 41.55 | Zone A 70.46% | A+B 97.74%

--------------------------------------------------------------------------------


In [6]:
def _extract(participant_result, metric):
    """Extract a metric from one participant's result."""
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

def build_summary(results_key, participant_key, participant_ids):
    """results_key/participant_key: ('val_results', 'val_results_per_participant') or
    ('test_results', 'test_results_per_participant')."""
    summary = {}
    for h in HORIZONS:
        summary[h] = {}
        for m in METRICS:
            if m == 'zone_a':
                seed_values = [r[results_key][h]['ceg']['A'] for r in all_results]
            elif m == 'zone_ab':
                seed_values = [r[results_key][h]['ceg']['A'] + r[results_key][h]['ceg']['B']
                                for r in all_results]
            else:
                seed_values = [r[results_key][h][m] for r in all_results]

            mean = float(np.mean(seed_values))
            sd1 = float(np.std(seed_values, ddof=1))

            participant_means = []
            for pid in participant_ids:
                values_across_seeds = [_extract(r[participant_key][pid][h], m)
                                        for r in all_results]
                participant_means.append(np.mean(values_across_seeds))
            sd2 = float(np.std(participant_means, ddof=1))

            summary[h][m] = {
                'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2,
            }
    return summary

In [7]:
val_summary = build_summary('val_results', 'val_results_per_participant', VAL_PARTICIPANT_IDS)
test_summary = build_summary('test_results', 'test_results_per_participant', PARTICIPANT_IDS)

In [8]:
def _print_summary(split_label, summary):
    print(f'=== {MODEL_NAME} {split_label} set: overall mean +/- SD_seed (SD_participant) ===')
    for h in HORIZONS:
        s = summary[h]
        print(
            f"{h:>3}-min:  "
            f"RMSE = {s['rmse']['mean']:.2f} +/- "
            f"{s['rmse']['sd1_across_seeds']:.2f} "
            f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
            f"MAE = {s['mae']['mean']:.2f} +/- "
            f"{s['mae']['sd1_across_seeds']:.2f} "
            f"({s['mae']['sd2_across_participants']:.2f})   "
            f"gRMSE = {s['grmse']['mean']:.2f} +/- "
            f"{s['grmse']['sd1_across_seeds']:.2f} "
            f"({s['grmse']['sd2_across_participants']:.2f})   "
            f"Zone A = {s['zone_a']['mean']:.2f} +/- "
            f"{s['zone_a']['sd1_across_seeds']:.2f} "
            f"({s['zone_a']['sd2_across_participants']:.2f})%   "
            f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- "
            f"{s['zone_ab']['sd1_across_seeds']:.2f} "
            f"({s['zone_ab']['sd2_across_participants']:.2f})%"
        )

_print_summary('validation', val_summary)

=== CGM+time+dinov2 (gated) validation set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 12.26 +/- 0.69 (2.58) mg/dL   MAE = 8.35 +/- 0.47 (1.75)   gRMSE = 13.56 +/- 0.80 (3.65)   Zone A = 95.31 +/- 0.95 (4.49)%   Zone A+B = 99.56 +/- 0.10 (0.81)%
 30-min:  RMSE = 18.38 +/- 0.25 (4.35) mg/dL   MAE = 12.25 +/- 0.18 (2.83)   gRMSE = 21.04 +/- 0.34 (6.58)   Zone A = 88.73 +/- 0.31 (5.76)%   Zone A+B = 99.22 +/- 0.01 (1.11)%
 60-min:  RMSE = 26.44 +/- 0.26 (7.99) mg/dL   MAE = 17.36 +/- 0.14 (5.31)   gRMSE = 31.54 +/- 0.29 (12.37)   Zone A = 79.88 +/- 0.42 (6.53)%   Zone A+B = 98.59 +/- 0.09 (1.48)%
 90-min:  RMSE = 31.24 +/- 0.20 (10.97) mg/dL   MAE = 20.71 +/- 0.15 (7.23)   gRMSE = 38.06 +/- 0.22 (16.84)   Zone A = 74.53 +/- 0.48 (8.75)%   Zone A+B = 98.07 +/- 0.06 (2.06)%
120-min:  RMSE = 33.88 +/- 0.30 (12.42) mg/dL   MAE = 22.94 +/- 0.22 (8.18)   gRMSE = 41.58 +/- 0.38 (19.10)   Zone A = 70.62 +/- 0.62 (9.31)%   Zone A+B = 97.76 +/- 0.05 (2.53)%


In [9]:
with open(VAL_STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': VAL_PARTICIPANT_IDS,
        'per_horizon_summary': val_summary,
    }, f, indent=2)

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seeds': SEEDS,
        'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS,
        'per_horizon_summary': test_summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}/lstm_seed*_results.json')
print(f'saved validation (selection) summary to {VAL_STABILITY_SUMMARY_PATH}')
print(f'saved test (confirmatory-only) summary to {STABILITY_SUMMARY_PATH}')


saved per-seed results to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_dinov2/lstm_seed*_results.json
saved validation (selection) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_dinov2/gated_cgm_time_dinov2_val_stability_summary.json
saved test (confirmatory-only) summary to /content/drive/MyDrive/dissertation/experiments/3_initial_multimodal/multimodal/results/gated_dinov2/gated_cgm_time_dinov2_stability_summary.json
